In [1]:
# Cell 1: GPU check and Google Drive mount
import torch
from google.colab import drive

print("GPU available:", torch.cuda.is_available())
print("Name:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

GPU available: True
Name: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Cell 2: Python version and TerraTorch installation
import sys
print("Python:", sys.version)

# TerraTorch provides the Prithvi backbones, the UperNet decoder and the model factory
!pip install -q terratorch rasterio

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
# Cell 3: Import check and library versions (versions read from the installed package records)
from importlib.metadata import version
import torch
import terratorch

print("TerraTorch:", version("terratorch"))
print("PyTorch:", torch.__version__)
print("numpy:", version("numpy"))
print("CUDA available:", torch.cuda.is_available())

TerraTorch: 1.2.13
PyTorch: 2.11.0+cu130
numpy: 2.5.3
CUDA available: True


In [5]:
# Cell 4: Normalisation values in the official TerraTorch Sen1Floods11 data module
import inspect
from terratorch.datamodules import sen1floods11 as sen1floods11_dm

source_lines = inspect.getsource(sen1floods11_dm).splitlines()
for number, line in enumerate(source_lines, start=1):
    if any(word in line.lower() for word in ["mean", "std", "bands", "constant_scale"]):
        print(f"{number:4d}: {line}")

  16: MEANS = {
  32: STDS = {
  57:         bands: Sequence[str] = Sen1Floods11NonGeo.all_band_names,
  63:         constant_scale: float = 0.0001,
  76:             bands (Sequence[str], optional): List of bands to use. Defaults to Sen1Floods11NonGeo.all_band_names.
  82:             constant_scale (float, optional): Scale constant applied to the dataset. Defaults to 0.0001.
  91:         means = [MEANS[b] for b in bands]
  92:         stds = [STDS[b] for b in bands]
  93:         self.bands = bands
  98:         self.aug = AugmentationSequential(K.Normalize(means, stds), data_keys=None)
 100:         self.constant_scale = constant_scale
 116:                 bands=self.bands,
 117:                 constant_scale=self.constant_scale,
 127:                 bands=self.bands,
 128:                 constant_scale=self.constant_scale,
 138:                 bands=self.bands,
 139:                 constant_scale=self.constant_scale,
 149:                 bands=self.bands,
 150:             

In [3]:
# Cell 5: Prithvi-EO-2.0-300M-TL with the official Sen1Floods11 settings; GPU memory and speed test
import time
import torch.nn as nn
from terratorch.models import EncoderDecoderFactory

device = torch.device("cuda")
BANDS = ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"]

def build_prithvi(freeze_backbone):
    """Prithvi-EO-2.0-300M-TL backbone (pretrained) with an UperNet decoder, as in the official config."""
    model = EncoderDecoderFactory().build_model(
        task="segmentation",
        backbone="prithvi_eo_v2_300_tl",
        backbone_pretrained=True,
        backbone_bands=BANDS,
        necks=[{"name": "SelectIndices", "indices": [5, 11, 17, 23]},
               {"name": "ReshapeTokensToImage"}],
        decoder="UperNetDecoder",
        decoder_channels=256,
        decoder_scale_modules=True,
        head_dropout=0.1,
        num_classes=2,
        rescale=True,
    )
    if freeze_backbone:
        for p in model.encoder.parameters():
            p.requires_grad = False
    return model

ce_loss = nn.CrossEntropyLoss(ignore_index=-1)

for name, freeze in [("frozen", True), ("full", False)]:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = build_prithvi(freeze).to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5, weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda")

    # Three training steps on random input (batch 16, 6 bands, 224 x 224) and random labels
    x = torch.randn(16, 6, 224, 224, device=device)
    y = torch.randint(0, 2, (16, 224, 224), device=device)
    step_times = []
    for step in range(3):
        t0 = time.time()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(x)
        logits = out.output if hasattr(out, "output") else out
        loss = ce_loss(logits.float(), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()
        step_times.append(time.time() - t0)

    print(f"{name}: parameters = {total / 1e6:.0f} M, trainable = {trainable / 1e6:.1f} M")
    print(f"    output shape = {tuple(logits.shape)}")
    print(f"    peak GPU memory = {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    print(f"    step time (after warm-up) = {sum(step_times[1:]) / 2:.2f} s")

    del model, optimizer

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


TypeError: UperNetDecoder.__init__() got an unexpected keyword argument 'scale_modules'

In [4]:
# Cell 5: Prithvi-EO-2.0-300M-TL with the TerraTorch 1.x Sen1Floods11 example settings; GPU memory and speed test
import time
import torch.nn as nn
from terratorch.models import EncoderDecoderFactory

device = torch.device("cuda")
BANDS = ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"]
BATCH = 8

def build_prithvi(freeze_backbone):
    """Prithvi-EO-2.0-300M-TL backbone (pretrained, 512 x 512 input) with a UNet decoder, as in the example config."""
    model = EncoderDecoderFactory().build_model(
        task="segmentation",
        backbone="prithvi_eo_v2_300_tl",
        backbone_pretrained=True,
        backbone_img_size=512,
        backbone_coords_encoding=[],
        backbone_bands=BANDS,
        necks=[{"name": "SelectIndices", "indices": [5, 11, 17, 23]},
               {"name": "ReshapeTokensToImage"},
               {"name": "LearnedInterpolateToPyramidal"}],
        decoder="UNetDecoder",
        decoder_channels=[512, 256, 128, 64],
        head_dropout=0.1,
        num_classes=2,
    )
    if freeze_backbone:
        for p in model.encoder.parameters():
            p.requires_grad = False
    return model

ce_loss = nn.CrossEntropyLoss(ignore_index=-1)

for name, freeze in [("frozen", True), ("full", False)]:
    model, optimizer = None, None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        model = build_prithvi(freeze).to(device)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.1)
        scaler = torch.amp.GradScaler("cuda")

        # Three training steps on random input (batch 8, 6 bands, 512 x 512) and random labels
        x = torch.randn(BATCH, 6, 512, 512, device=device)
        y = torch.randint(0, 2, (BATCH, 512, 512), device=device)
        step_times = []
        for step in range(3):
            t0 = time.time()
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                out = model(x)
            logits = out.output if hasattr(out, "output") else out
            loss = ce_loss(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            torch.cuda.synchronize()
            step_times.append(time.time() - t0)

        print(f"{name}: parameters = {total / 1e6:.0f} M, trainable = {trainable / 1e6:.1f} M")
        print(f"    output shape = {tuple(logits.shape)}")
        print(f"    peak GPU memory = {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
        print(f"    step time (after warm-up) = {sum(step_times[1:]) / 2:.2f} s, "
              f"estimated epoch (31 steps) = {31 * sum(step_times[1:]) / 2:.0f} s")
    except torch.cuda.OutOfMemoryError:
        print(f"{name}: OUT OF GPU MEMORY at batch {BATCH} and 512 x 512")
    finally:
        del model, optimizer
        torch.cuda.empty_cache()

frozen: parameters = 324 M, trainable = 20.3 M
    output shape = (8, 2, 512, 512)
    peak GPU memory = 2.8 GB
    step time (after warm-up) = 0.55 s, estimated epoch (31 steps) = 17 s
full: parameters = 324 M, trainable = 324.2 M
    output shape = (8, 2, 512, 512)
    peak GPU memory = 14.5 GB
    step time (after warm-up) = 1.64 s, estimated epoch (31 steps) = 51 s


### Part C

In [5]:
# Cell 1 (new first lines): PyTorch memory allocator setting that reduces GPU memory fragmentation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [6]:
# Cell 6: Repository update and package import path
import sys
from pathlib import Path

REPO_URL = "https://github.com/Prhm93/floodfm-sen1floods11.git"
REPO_DIR = Path("/content/floodfm-sen1floods11")

# Clone on the first run in a session; pull the latest commits on later runs
if REPO_DIR.exists():
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# The src folder is added to the Python search path, so that "import floodfm" finds the package
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

Cloning into '/content/floodfm-sen1floods11'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 64 (delta 24), reused 55 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 4.90 MiB | 9.52 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [7]:
# Cell 7: Dataset copy from Google Drive to the local Colab disk (skipped when the local copy is complete)
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/data/sen1floods11")
LOCAL_ROOT = Path("/content/data/sen1floods11")
FOLDERS = ["HandLabeled/S1Hand", "HandLabeled/S2Hand", "HandLabeled/LabelHand", "splits/flood_handlabeled"]

def local_copy_complete():
    """Returns True if each of the three layer folders on the local disk contains 446 .tif files."""
    return all(len(list((LOCAL_ROOT / f).glob("*.tif"))) == 446 for f in FOLDERS[:3])

if local_copy_complete():
    print("Local copy already complete.")
else:
    t0 = time.time()
    for folder in FOLDERS:
        shutil.copytree(DRIVE_ROOT / folder, LOCAL_ROOT / folder, dirs_exist_ok=True)
    print(f"Copy time: {time.time() - t0:.0f} s")

for folder in FOLDERS:
    print(f"{folder}: {len(list((LOCAL_ROOT / folder).iterdir()))} files")

Copy time: 657 s
HandLabeled/S1Hand: 446 files
HandLabeled/S2Hand: 446 files
HandLabeled/LabelHand: 446 files
splits/flood_handlabeled: 4 files


In [8]:
# Cell 8: Short test run (frozen, seed 0, 2 epochs) in the separate test folder
import importlib
import floodfm.prithvi
importlib.reload(floodfm.prithvi)
from floodfm.prithvi import train_prithvi

TEST_ROOT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/checkpoints/_test")
history = train_prithvi(LOCAL_ROOT, TEST_ROOT, mode="frozen", seed=0, epochs=2)

epoch   1 | loss 0.3965 | valid loss 0.3058 | valid IoU 0.6315 | best 0.6315 | lr 1.0e-04 | 30.6 s
epoch   2 | loss 0.3418 | valid loss 0.2766 | valid IoU 0.6256 | best 0.6315 | lr 1.0e-04 | 33.8 s


In [ ]:
# Cell 9: Three frozen Prithvi runs (seeds 0, 1, 2): training, then evaluation; safe to re-run after a disconnect
import subprocess
importlib.reload(floodfm.prithvi)
from floodfm.prithvi import train_prithvi, evaluate_prithvi

PROJECT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")
CKPT_ROOT = PROJECT / "checkpoints"
RESULTS_DIR = PROJECT / "results"
PRED_DIR = PROJECT / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 100
MODE = "frozen"

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

for seed in [0, 1, 2]:
    method = f"prithvi_{MODE}_seed{seed}"
    results_path = RESULTS_DIR / f"{method}.json"
    if results_path.exists():
        print(f"{method}: results file exists, skipped")
        continue

    print(f"\n===== {method}: training =====")
    train_prithvi(LOCAL_ROOT, CKPT_ROOT, mode=MODE, seed=seed, epochs=EPOCHS)

    print(f"===== {method}: evaluation =====")
    res = evaluate_prithvi(
        LOCAL_ROOT, CKPT_ROOT / method / "best.pt", MODE, method,
        description=(f"Prithvi-EO-2.0-300M-TL (TerraTorch), backbone {MODE}, UNet decoder, 6 Sentinel-2 bands, "
                     f"seed {seed}, {EPOCHS} epochs, best validation epoch; settings from the TerraTorch 1.x "
                     f"Sen1Floods11 example config; pixels without Sentinel-2 image predicted as not water"),
        results_path=results_path, prediction_path=PRED_DIR / f"{method}_bolivia.npz", code_commit=commit,
    )
    for split in ["test", "bolivia"]:
        s = res["splits"][split]["overall"]
        print(f"{method} | {split}: IoU = {s['iou']:.3f}, F1 = {s['f1']:.3f}")


===== prithvi_frozen_seed0: training =====
